# Sequence Classification with Conv1D

Classify synthetic waveforms (sine, square, triangle) using 1D convolutions.
This example demonstrates Conv1D, MaxPool1D, dropout, and gradient accumulation.

```
Conv1D(1->4, k=3) -> ReLU -> MaxPool1D(2) ->
Conv1D(4->8, k=3) -> ReLU -> MaxPool1D(2) ->
Dropout(0.5) -> Linear(48->3) (raw logits; loss applies log_softmax)
```

**CLI equivalent:** `make example-seq-classify` (1000 epochs)


## Architecture

The same type-safe dimension chain as the CNN notebook, but in 1D.
`ConvOutDim` and `PoolOutDim` work identically for 1D and 2D.


In [ ]:
:t conv1dLayer


In [ ]:
:t maxPool1dLayer


In [ ]:
:t dropoutLayer


## Dimension Chain

Input: 32 timesteps, 1 channel (flat dim = 32).

| Layer | Length | Channels | Flat dim |
|-------|--------|----------|----------|
| Input | 32 | 1 | 32 |
| Conv1D(k=3) | 30 | 4 | 120 |
| MaxPool(2) | 15 | 4 | 60 |
| Conv1D(k=3) | 13 | 8 | 104 |
| MaxPool(2) | 6 | 8 | 48 |
| Dropout(0.5) | - | - | 48 |
| Linear | - | - | 3 |

All flat dims stay under 120 to avoid Idris 2's Peano Nat
type-checking ceiling (~1000).


## Data: Synthetic Waveforms

Three classes with random frequency and phase:
- **Sine**: smooth oscillation
- **Square**: binary high/low
- **Triangle**: linear ramps

Fresh data is generated each epoch, so the model must learn general
waveform features rather than memorizing specific samples.


## Model Construction

We can build and inspect the full Conv1D pipeline interactively.
The waveform data generator is in the compiled example
(`src/Example/SeqClassify.idr`).


In [ ]:
:exec do { srand 42;
  conv1 <- conv1dLayer {inC=1, outC=4, len=32, kL=3, pad=0};
  conv2 <- conv1dLayer {inC=4, outC=8, len=15, kL=3, pad=0};
  fc <- linearLayerAny {i=48} {o=3} "ll0";
  model <- pure ((
    conv1 ~~> reluLayerAny ~~> maxPool1dLayer {c=4, len=30, poolK=2, str=2}
    ~~> conv2 ~~> reluLayerAny ~~> maxPool1dLayer {c=8, len=13, poolK=2, str=2}
    ~~> dropoutLayer 0.5
    ~~> OutputLayer fc));
  putStrLn "SeqClassify model built successfully (show unavailable in REPL for large dims)";
  putStrLn ("Param count: " ++ show (networkParamCount model)) }


> **Note:** The cell above may produce a type constraint error in the REPL due to Idris 2's Peano Nat reduction limits at large dimensions. The CLI example (`make example-seq-classify`) compiles and runs correctly. See `docs/develop/gotchas.md` for details.


Training requires the waveform data generator (defined in
`src/Example/SeqClassify.idr`). The training loop:

```idris
opt <- pure (nativeAdamGlobalClip 0.001 0.9 0.999 1.0e-8 1.0)
(trained, epochs, loss) <- runTraining
  (\m, d => epochNativeTensorPre opt d seqCE m)
  (seqBatch 32) (simpleConfig 1000) model
```

Fresh waveforms are generated each epoch, so the model learns general
features rather than memorizing samples.

Run via CLI: `make example-seq-classify --epochs 1000`


## PyTorch Comparison

```python
model = nn.Sequential(
    nn.Conv1d(1, 4, 3), nn.ReLU(), nn.MaxPool1d(2),
    nn.Conv1d(4, 8, 3), nn.ReLU(), nn.MaxPool1d(2),
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(48, 3)
)
```

Note that PyTorch needs `nn.Flatten()` between the conv and linear layers.
In idris-ml, the flat dimension `48 = 8 * 6` is already a type-level Nat,
so no flatten is needed (or possible to get wrong).

See `pytorch/torch_ref/scripts/seq_classify.py` for the full reference.
